# SGP Kits Statistics Report

This notebook analyzes the normalized kit statistics extracted from `command_storage.dat`.

The main focus is kit-level performance. Player IDs are kept as an explanatory dimension so we can detect cases where a kit's aggregate kill count is heavily driven by one or a few players.

## Setup

The companion `sgp_report.py` module handles data loading, validation, shared aggregations, and the chart patterns reused across sections. This keeps the notebook focused on the analysis itself.


In [ ]:
from pathlib import Path

import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sgp_report import (
    KIT_NAMES,
    KIT_ORDER,
    concentration_figure,
    load_report_data,
    player_contribution_figure,
    relative_metric_heatmap,
    show_player_contribution_figure,
)


report = load_report_data(Path("data"))


## Total kills by kit

Switch between total kills per kit and the same totals stacked by player contribution. Player IDs appear on hover in the stacked view. Click a player segment to emphasize that player across every bar; click the same player again to restore all colors.

In [ ]:
fig = player_contribution_figure(
    all_kits=report.all_kits,
    totals=report.total_kills_by_kit,
    by_player=report.player_kit_kills,
    player_col="id_killer",
    value_col="kills",
    title="Total kills by kit",
    yaxis_title="Kills",
    total_button_label="Total kills",
)
show_player_contribution_figure(fig)


## Player concentration of kills

The top-player and top-3 shares show whether a kit's aggregate kill count is broadly distributed or dominated by a small number of players.

In [ ]:
fig = concentration_figure(
    report.kit_kill_stats,
    total_col="kills",
    top_player_col="top_player_share",
    top_three_col="top_3_share",
    title="Kill concentration by kit",
    yaxis_title="Share of kit kills",
)
fig.show()


## Total kills vs. player concentration

Each point is a kit. This separates high-kill kits with broad contribution from high-kill kits whose result is driven mostly by one player.

In [ ]:
scatter_data = report.kit_kill_stats.loc[
    report.kit_kill_stats["kills"] > 0,
    [
        "kit_name",
        "kills",
        "players",
        "top_player_share",
        "top_3_share",
    ],
].copy()

fig = px.scatter(
    scatter_data,
    x="kills",
    y="top_player_share",
    text="kit_name",
    hover_name="kit_name",
    hover_data={
        "kills": True,
        "players": True,
        "top_player_share": ":.1%",
        "top_3_share": ":.1%",
        "kit_name": False,
    },
    labels={
        "kills": "Total kills",
        "top_player_share": "Top player's share of kit kills",
        "players": "Players with kills",
        "top_3_share": "Top 3 players' share",
    },
    title="Total kills vs. player concentration",
)

fig.update_traces(textposition="top center")
fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()


# Matchups

These plots describe observed kill counts between kits. They are not win-rate estimates because the dataset does not contain the number of encounters or time played in each matchup.

## Kill matrix

Rows are killer kits and columns are victim kits. This is the direct view of where each kit's kills came from.

In [ ]:
matchup_matrix = report.matchup_matrix

fig = go.Figure(
    go.Heatmap(
        z=matchup_matrix.values,
        x=KIT_ORDER,
        y=KIT_ORDER,
        text=matchup_matrix.values,
        texttemplate="%{text}",
        hovertemplate=(
            "<b>%{y} → %{x}</b><br>"
            "Kills: %{z}<extra></extra>"
        ),
        colorbar=dict(title="Kills"),
    )
)

fig.update_layout(
    title="Kills by killer kit and victim kit",
    xaxis_title="Victim kit",
    yaxis_title="Killer kit",
)
fig.show()


## Directional kill share

For each pair of kits, this shows the share of observed kills going in one direction. A value above 50% means the row kit killed the column kit more often than the reverse.

This is useful for spotting asymmetric observed matchups, but it is still **not a win rate**. Hover also shows the total number of kills observed between the two kits, so low-volume extremes are easy to identify.

In [ ]:
directional_share = report.directional_share
pair_totals = report.pair_totals
hover_text = np.empty_like(directional_share, dtype=object)

for i, row_kit in enumerate(KIT_NAMES):
    for j, column_kit in enumerate(KIT_NAMES):
        if i == j:
            hover_text[i, j] = f"{row_kit} vs itself"
        elif np.isnan(directional_share[i, j]):
            hover_text[i, j] = (
                f"{row_kit} → {column_kit}<br>"
                "No kills observed in either direction"
            )
        else:
            hover_text[i, j] = (
                f"{row_kit} → {column_kit}<br>"
                f"Directional share: {directional_share[i, j]:.1%}<br>"
                f"Pair kills observed: {pair_totals[i, j]}"
            )

fig = go.Figure(
    go.Heatmap(
        z=directional_share,
        x=KIT_ORDER,
        y=KIT_ORDER,
        zmin=0,
        zmax=1,
        zmid=0.5,
        customdata=hover_text,
        hovertemplate="%{customdata}<extra></extra>",
        colorbar=dict(title="Share", tickformat=".0%"),
    )
)

fig.update_layout(
    title="Directional share of observed kills between kit pairs",
    xaxis_title="Other kit",
    yaxis_title="Row kit",
)
fig.show()


# Ability usage

Ability-use counts are analyzed in the same spirit as kills: first by total volume, then by how concentrated that volume is among players.

## Ability uses by kit

Switch between total uses and player-stacked contributions. Player IDs appear on hover. Click a player segment to emphasize that player across every bar; click the same player again to restore all colors.

In [ ]:
fig = player_contribution_figure(
    all_kits=report.all_kits,
    totals=report.total_abilities_by_kit,
    by_player=report.player_kit_abilities,
    player_col="id",
    value_col="ability_use",
    title="Ability uses by kit",
    yaxis_title="Ability uses",
    total_button_label="Total uses",
)
show_player_contribution_figure(fig)


## Player concentration of ability usage

This shows whether a kit's total ability usage is broadly distributed or mainly produced by one or a few players.

In [ ]:
fig = concentration_figure(
    report.kit_ability_stats,
    total_col="ability_use",
    top_player_col="top_player_ability_share",
    top_three_col="top_3_ability_share",
    title="Ability-use concentration by kit",
    yaxis_title="Share of ability uses",
)
fig.show()


# Combined analysis

These plots combine the two types of evidence without assuming that ability uses and kills are directly comparable measures of strength.

## Proportion of players who tried each kit

The denominator is every player appearing anywhere in the extracted kill or ability-use data.

Two signals are shown separately:

- players who used the kit's ability at least once;
- players who made at least one kill with the kit.

They are intentionally not merged into a single definition of “tried”, because each signal can miss some genuine kit usage.

In [ ]:
reach = report.reach
n_players = report.n_players

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=reach["kit_name"],
        y=reach["used_ability"],
        name="Used ability ≥ 1 time",
        customdata=np.column_stack([
            reach["used_ability_count"],
            np.full(len(reach), n_players),
        ]),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Used ability: %{y:.1%}<br>"
            "Players: %{customdata[0]:.0f} / %{customdata[1]:.0f}"
            "<extra></extra>"
        ),
    )
)

fig.add_trace(
    go.Bar(
        x=reach["kit_name"],
        y=reach["made_kill"],
        name="Made ≥ 1 kill",
        customdata=np.column_stack([
            reach["made_kill_count"],
            np.full(len(reach), n_players),
        ]),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Made a kill: %{y:.1%}<br>"
            "Players: %{customdata[0]:.0f} / %{customdata[1]:.0f}"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title="Proportion of observed players who tried each kit",
    xaxis_title="Kit",
    yaxis_title="Proportion of players",
    barmode="group",
)

fig.update_yaxes(tickformat=".0%", range=[0, 1])
fig.show()


## Kills vs. ability uses

Each point is a kit. This is a descriptive comparison of aggregate kill volume and aggregate ability-use volume; it does not imply that one causes the other or that their ratio has the same meaning across kits.

In [ ]:
combined_totals = report.combined_totals

fig = px.scatter(
    combined_totals,
    x="ability_use",
    y="kills",
    text="kit_name",
    hover_name="kit_name",
    hover_data={
        "ability_use": True,
        "kills": True,
        "kit_name": False,
    },
    labels={
        "ability_use": "Ability uses",
        "kills": "Kills",
    },
    title="Kills vs. ability uses by kit",
)

fig.update_traces(textposition="top center")
fig.show()


# Summary

The final plot gives a compact cross-metric profile of every kit. Each row is normalized independently, so color intensity means “high relative to the other kits for this metric”, not that different metrics share a common unit.

In [ ]:
metric_specs = [
    ("kills", "Total kills", "count"),
    ("ability_use", "Ability uses", "count"),
    ("made_kill", "Players with ≥1 kill", "percent"),
    ("used_ability", "Players using ability", "percent"),
    ("top_player_share", "Top-player kill share", "percent"),
    ("top_player_ability_share", "Top-player ability-use share", "percent"),
]

fig = relative_metric_heatmap(
    report.summary,
    metric_specs,
    title="Kit profile across the main report metrics",
)
fig.show()
